# 29. Vanilla co-training ablation with full Optuna tuning

Runs per-experiment Optuna tuning for vanilla co-training (Blum & Mitchell 1998).
No external pseudo-labels - models teach each other iteratively.

**Search space (7 params)**:
- lr: 1e-5 to 1e-3 (log)
- batch_size: [8, 16, 32, 64]
- train_epochs: 3 to 10 (epochs per iteration)
- samples_per_class: [1, 5, 10] (samples exchanged per class per iteration)
- finetune_patience: 4 to 10
- weight_decay: 0.0 to 0.1
- warmup_ratio: 0.0 to 0.3

**Fixed**: num_iterations=30 (capped by unlabeled pool size)

## Results path

```
results/bertweet/ablation/vanilla-cotrain/optuna/
```

## Setup

In [ ]:
import sys
import os
import time
import json
import subprocess
import warnings
from pathlib import Path
from collections import defaultdict

os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*emoji is not installed.*")
warnings.filterwarnings("ignore", message=".*not sharded.*")

import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

def _find_repo_root(marker: str = "lg_cotrain") -> Path:
    for candidate in [Path().resolve()] + list(Path().resolve().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Cannot find repo root: no ancestor directory contains '{marker}/'")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Python:    {sys.version.split()[0]}")

In [ ]:
import torch
print(f"torch version:     {torch.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"CUDA device count: {N_GPUS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    free_gb, total_gb = (x / 1024**3 for x in torch.cuda.mem_get_info(i))
    print(f"  cuda:{i}  {props.name}  total={total_gb:.1f} GB  free={free_gb:.1f} GB")

In [ ]:
EVENTS = [
    "kaikoura_earthquake_2016", "canada_wildfires_2016", "cyclone_idai_2019",
    "hurricane_florence_2018", "hurricane_maria_2017", "california_wildfires_2018",
    "hurricane_dorian_2019", "kerala_floods_2018", "hurricane_harvey_2017",
    "hurricane_irma_2017",
]
BUDGETS = [5, 10, 25, 50]
SEEDS = [1, 2, 3]
MODEL_NAME = "vinai/bertweet-base"
DATA_ROOT = repo_root / "data"
N_OPTUNA_TRIALS = 10

OPTUNA_STORAGE = repo_root / "results" / "bertweet" / "ablation" / "vanilla-cotrain" / "optuna"

# Reference results
LG_COTRAIN_OPTUNA = repo_root / "results" / "bertweet" / "optuna" / "wge7"
SELF_TRAINED_OPTUNA = repo_root / "results" / "bertweet" / "ablation" / "self-trained" / "optuna"
SUP_OPTUNA_DIR = repo_root / "results" / "bertweet" / "supervised" / "optuna-tuned"

print(f"Events:          {len(EVENTS)}")
print(f"Budgets:         {BUDGETS}")
print(f"Optuna trials:   {N_OPTUNA_TRIALS}")
print(f"Model:           {MODEL_NAME}")
print(f"Num GPUs:        {N_GPUS}")
print(f"Optuna storage:  {OPTUNA_STORAGE}")

In [ ]:
def run_optuna(budgets_to_run):
    """Run vanilla co-training Optuna per-experiment.
    
    Tunes 7 params: lr, batch_size, train_epochs, samples_per_class,
    finetune_patience, weight_decay, warmup_ratio.
    Fixed: num_iterations=30.
    """
    cmd = [
        sys.executable, "-m", "vanilla_cotrain.optuna_tuner",
        "--n-trials", str(N_OPTUNA_TRIALS),
        "--num-gpus", str(N_GPUS),
        "--events", *EVENTS,
        "--budgets", *(str(b) for b in budgets_to_run),
        "--seed-sets", *(str(s) for s in SEEDS),
        "--model-name", MODEL_NAME,
        "--data-root", str(DATA_ROOT),
        "--storage-dir", str(OPTUNA_STORAGE),
    ]
    n_studies = len(EVENTS) * len(budgets_to_run) * len(SEEDS)
    print(f"Budgets: {budgets_to_run}")
    print(f"Expected: {n_studies} studies x {N_OPTUNA_TRIALS} trials = {n_studies * N_OPTUNA_TRIALS} runs")
    print("=" * 70)

    start = time.time()
    proc = subprocess.run(cmd, cwd=str(repo_root))
    elapsed = time.time() - start

    print("=" * 70)
    print(f"Wall clock: {elapsed:.0f}s = {elapsed/3600:.2f}h")
    if proc.returncode != 0:
        print(f"WARNING: exit code {proc.returncode}")

    n_done = len(list(OPTUNA_STORAGE.glob(f"*/*/trials_{N_OPTUNA_TRIALS}/best_params.json")))
    total_expected = len(EVENTS) * len(BUDGETS) * len(SEEDS)
    print(f"\nbest_params.json files: {n_done} / {total_expected} (all budgets)")


def analyze(budgets_to_show=None):
    """Compare vanilla Optuna vs LG-CoTrain vs self-trained vs supervised."""
    if budgets_to_show is None:
        budgets_to_show = BUDGETS

    sources = {
        "LG-CoTrain (GPT-4o)": LG_COTRAIN_OPTUNA,
        "Self-trained (NB27)": SELF_TRAINED_OPTUNA,
        "Vanilla (this NB)": OPTUNA_STORAGE,
        "Supervised": SUP_OPTUNA_DIR,
    }

    data = {}
    for label, path in sources.items():
        data[label] = defaultdict(list)
        for f in sorted(path.glob(f"*/*/trials_{N_OPTUNA_TRIALS}/best_params.json")):
            p = json.loads(f.read_text(encoding="utf-8"))
            fm = p.get("best_full_metrics", {})
            if fm and fm["budget"] in budgets_to_show:
                data[label][fm["budget"]].append(fm["test_macro_f1"])

    print("=" * 100)
    print("COMPARISON: Vanilla vs Self-trained vs LG-CoTrain vs Supervised")
    print("=" * 100)
    header = f'{"Budget":>8s}'
    for label in sources:
        header += f'  {label:>22s}'
    print(header)
    print("-" * 100)

    for b in budgets_to_show:
        row = f"{b:>8d}"
        for label in sources:
            vals = data[label].get(b, [])
            if vals:
                row += f"  {sum(vals)/len(vals):>18.4f} ({len(vals):>2d})"
            else:
                row += f"  {'n/a':>22s}"
        print(row)

    # Per-event breakdown
    print()
    for b in budgets_to_show:
        van_by_event = defaultdict(list)
        lg_by_event = defaultdict(list)
        for f in sorted(OPTUNA_STORAGE.glob(f"*/{b}_set*/trials_{N_OPTUNA_TRIALS}/best_params.json")):
            p = json.loads(f.read_text(encoding="utf-8"))
            fm = p.get("best_full_metrics", {})
            if fm:
                van_by_event[fm["event"]].append(fm["test_macro_f1"])
                # Also show best params chosen
        for f in sorted(LG_COTRAIN_OPTUNA.glob(f"*/{b}_set*/trials_{N_OPTUNA_TRIALS}/best_params.json")):
            p = json.loads(f.read_text(encoding="utf-8"))
            fm = p.get("best_full_metrics", {})
            if fm:
                lg_by_event[fm["event"]].append(fm["test_macro_f1"])

        if not van_by_event:
            continue

        print(f"--- Budget {b} per-event ---")
        print(f'{"Event":<35s}  {"LG-CoTrain":>10s}  {"Vanilla":>10s}  {"Delta":>8s}')
        for event in EVENTS:
            lg_v = lg_by_event.get(event, [])
            van_v = van_by_event.get(event, [])
            lg_m = sum(lg_v)/len(lg_v) if lg_v else 0
            van_m = sum(van_v)/len(van_v) if van_v else 0
            lg_s = f"{lg_m:.4f}" if lg_v else "n/a"
            van_s = f"{van_m:.4f}" if van_v else "n/a"
            d_s = f"{van_m - lg_m:+.4f}" if lg_v and van_v else "n/a"
            print(f"{event:<35s}  {lg_s:>10s}  {van_s:>10s}  {d_s:>8s}")
        print()

## Budget 5

In [ ]:
run_optuna(budgets_to_run=[5])

In [ ]:
analyze(budgets_to_show=[5])

## Budget 10

In [ ]:
run_optuna(budgets_to_run=[10])

In [ ]:
analyze(budgets_to_show=[5, 10])

## Budget 25

In [ ]:
run_optuna(budgets_to_run=[25])

In [ ]:
analyze(budgets_to_show=[5, 10, 25])

## Budget 50

In [ ]:
run_optuna(budgets_to_run=[50])

In [ ]:
analyze(budgets_to_show=[5, 10, 25, 50])

## Summary

This notebook tunes vanilla co-training with 7 hyperparameters (including
samples_per_class and train_epochs), using the same per-experiment Optuna
approach as LG-CoTrain for a fair comparison.